<div align="center">
    <img src="https://www.sharif.ir/documents/20124/0/logo-fa-IR.png/4d9b72bc-494b-ed5a-d3bb-e7dfd319aec8?t=1609608338755" alt="Logo" width="200">
    <p><b>Sharif University of Technology</b></p>
    <p>Deep Learning Course, Dr. Soleymani</p>
    <p>Spring 2026</p>
</div>

---


*Full Name: Raya Namazi*


# Retrieval-Augmented Generation (RAG) – From Scratch to Conversation

## Overview

**Retrieval-Augmented Generation (RAG)** is a technique that enhances a language model’s output by first retrieving relevant information from a knowledge base, then conditioning the generation on that retrieved evidence.

Mathematically, given a query $x$, we want to model:

$
p(y \mid x) = \sum_{z \in \text{Top-}k(x)} p_{\text{retrieve}}(z \mid x) \; p_{\text{generate}}(y \mid x, z)
$

where $z$ is a retrieved document chunk, often treated as a hard selection after top‑k retrieval.

### Dataset used in this notebook

Instead of a manually written toy knowledge base, this version uses a small real subset of **SQuAD v1** from Hugging Face. SQuAD provides real Wikipedia-based contexts, questions, and gold answers. To keep the notebook lightweight, we only use a small validation subset.

### In this notebook you will:

1. **From scratch (no LangChain)**:
   - Load a small real QA dataset
   - Chunk real Wikipedia contexts
   - Create dense embeddings with SentenceTransformers
   - Build a FAISS index for fast nearest-neighbour search
   - Implement a retriever and a GPT‑2 generator
   - Assemble a complete, stateless RAG system

2. **With LangChain**:
   - Integrate memory for multi-turn conversations
   - Implement a **history-aware** retriever that rewrites follow-up questions

3. **With an encoder-decoder generator**:
   - Replace GPT‑2 with FLAN‑T5
   - Compare decoder-only RAG and encoder-decoder RAG on the same retrieved contexts

After completing this notebook, you will understand the core components of RAG, the need for memory in conversational systems, and the architectural difference between decoder-only and encoder-decoder generators.

**Dependencies**: Run the cell below to install required packages.

In [2]:
# Install necessary packages (if not already installed)
import sys
!{sys.executable} -m pip install --quiet torch transformers sentence-transformers faiss-cpu datasets langchain-community langchain-huggingface langchain-core
# We use:
# - datasets for loading a small real SQuAD subset
# - sentence-transformers for dense embeddings
# - faiss-cpu for vector search
# - transformers for GPT-2 and FLAN-T5 generators
# - langchain-community / langchain-huggingface for the vectorstore wrapper in the conversational part


In [3]:
import os
import json
import torch
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Part 1: Document Loading & Text Chunking

A RAG system needs a collection of documents to retrieve from. In real applications these are often large texts, so we must split them into manageable **chunks**. Each chunk is a unit that can be embedded and later retrieved.

In this notebook, we use a small real subset of **SQuAD v1** instead of a manually written toy text file. Each SQuAD example contains:

- a real Wikipedia paragraph as `context`,
- a `question`,
- one or more gold `answers`.

We use the contexts as the retrieval corpus and the questions/gold answers for testing.

**Why chunk?**
- Language models have a limited context window.
- Smaller chunks allow more precise retrieval; a long document may contain many topics.

**Chunking trade‑offs**:
- Too small → loss of surrounding context.
- Too large → retrieval may bring irrelevant detail, and generation may exceed token limits.

**Implementation**: We’ll implement a simple recursive character‑based splitter with user‑defined chunk size and overlap.

### Questions 1
1. What would happen if we used whole documents without chunking?  
2. How does the overlap parameter help preserve continuity between chunks?

1. The embedding model and the generators have a maximum input length. A long Wikipedia context would be cut off, so part of the document would never get embedded or seen by the model. A single vector for a whole document is forced to average many different topics. A query about one specific fact matches poorly because the relevant sentence is diluted by surrounding unrelated text. Smaller chunks produce focused vectors and sharper matches.


2. A fixed-size split may cut the text in the middle of a sentence, or separates a fact from the words that give it meaning. The overlap makes each chunk repeat the last few tokens of the previous one, so any information sitting on a boundary still appears complete in at least one chunk. This keeps sentences and local context intact, so the embeddings stay meaningful and the retriever can still find the answer even when it falls near a chunk edge.

In [4]:
def chunk_text(text, chunk_size=100, overlap=20):
    
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    step = chunk_size - overlap  
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += step
    return chunks


sample_text = "A RAG system needs a collection of documents to retrieve from. In real applications these are often large texts, so we must split them into manageable **chunks**. Each chunk is a unit that can be embedded and later retrieved." * 41
test_chunks = chunk_text(sample_text, chunk_size=100, overlap=20)

print("Number of chunks:", len(test_chunks))
print("First chunk length:", len(test_chunks[0]))


Number of chunks: 116
First chunk length: 100


In [5]:
# TODO: Load a small real SQuAD subset using `load_dataset`.
# Then extract unique contexts and create `chunks` using `chunk_text`.
# Expected output is kept below as a reference.

from datasets import load_dataset

squad = load_dataset("rajpurkar/squad", split="validation[:300]")
print(squad)

print("Example row:")
print(squad[0])
print("Number of QA examples:", len(squad))

qa_dataset = squad

unique_contexts = list(dict.fromkeys(squad["context"]))
print("Number of unique contexts:", len(unique_contexts))

chunks = []
for context in unique_contexts:
    chunks.extend(chunk_text(context, chunk_size=500, overlap=0))
print(f"Created {len(chunks)} chunks from real SQuAD contexts.")

for i in range(3):
    print(f"\nChunk {i}:")
    print(chunks[i][:500], "...")

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 300
})
Example row:
{'id': '56be4db0acb8001400a502ec', 'title': 'Super_Bowl_50', 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.', 'question': 'Which NFL team represented

## Part 2: Document Embeddings & Vector Index

To retrieve relevant chunks, we need to embed both the documents and the query into a common dense vector space. We use a pre‑trained sentence transformer to obtain **fixed‑size embeddings**.

**Dense retrieval** computes:

$
\text{sim}(q, d) = \cos(\phi(q), \phi(d)) \quad \text{or} \quad \|\phi(q) - \phi(d)\|_2
$

where $\phi$ is the embedding function.

We’ll store the embeddings in a **FAISS** index for fast approximate (or exact) nearest‑neighbour search.

### Questions 2
1. Explain the difference between sparse retrieval (e.g., TF‑IDF, BM25) and dense retrieval.  
2. Why is L2 distance often used instead of cosine similarity in FAISS `IndexFlatL2`? (Hint: embedding normalisation)

1. Sparse retrieval represents each text as a high-dimensional vector over the vocabulary, where most entries are zero and only the words actually present get a non-zero weight. Matching is based on exact word overlap, so it works well when the query and the document share the same terms but fails when they use different words for the same meaning. Dense retrieval instead encodes the text with a neural model into a low-dimensional vector where every entry is non-zero. Similarity is measured in this learned semantic space, so two passages can match even if they use different words, as long as their meaning is close.

2. If the embeddings are L2-normalised to unit length, then minimising L2 distance is equivalent to maximising cosine similarity, because ||a - b||² = 2 - 2·cos(a, b) for unit vectors.

In [ ]:
# TODO: Load `all-MiniLM-L6-v2` with SentenceTransformer.
# Encode all chunks into float32 dense embeddings.
# Expected output is kept below as a reference.

from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

chunk_embeddings = embed_model.encode(
    chunks, show_progress_bar=True, normalize_embeddings=True
).astype("float32")
print("Embeddings shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384


/tmp/ipykernel_31785/3947491537.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (30, 384)


In [7]:
# TODO: Build a FAISS IndexFlatL2 index.
# Add all chunk embeddings to the index.
# Expected output is kept below as a reference.

import faiss

embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(chunk_embeddings)

print(f"FAISS index contains {index.ntotal} vectors.")

FAISS index contains 30 vectors.


## Part 3: The Retriever

The retriever is responsible for, given a user query, finding the top‑k most relevant chunks.

Formally:

$
\text{Retrieve}(q, k) = \text{argtopk}_{d \in \mathcal{D}} \; \text{sim}(\phi(q), \phi(d))
$

where $\mathcal{D}$ is the set of all chunk embeddings.

### Question 3
How would you modify the retriever to use **maximum inner product search** (MIPS) instead of L2 distance? When would MIPS be preferred?

To use maximum inner product search, I would build the index with `faiss.IndexFlatIP` instead of `faiss.IndexFlatL2`. The search call stays the same, but now FAISS scores each vector by the dot product with the query and returns the largest scores instead of the smallest distances, so a higher score means more relevant. If I also normalise the chunk and query embeddings to unit length with `faiss.normalize_L2` before searching, the inner product becomes exactly the cosine similarity.

MIPS is preferred when the embedding model was trained with a dot-product or cosine objective, because then the scoring matches how the vectors were actually learned. It is also the natural choice when the magnitude of the vectors carries useful information that we don't want to throw away, and it is the standard option used by large-scale approximate search indexes. When the vectors are L2-normalised, MIPS and L2 distance give the same ranking, so the difference only really matters for non-normalised embeddings.

In [8]:
# TODO: Implement `retrieve(query, index, embed_model, chunks, top_k)`.
# Embed the query, search FAISS, and return top-k (chunk, distance) pairs.
# Expected output is kept below as a reference.

def retrieve(query, index, embed_model, chunks, top_k=3):
    
    query_embedding = embed_model.encode(
        [query], normalize_embeddings=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append((chunks[idx], float(dist)))
    return results

sample = qa_dataset[0]
query = sample["question"]
gold = sample["answers"]["text"][0]

print("Question:", query)
print("Gold answer:", gold)

print("\nTop retrieved chunks:")
for chunk, dist in retrieve(query, index, embed_model, chunks, top_k=3):
    print(f"\nDistance {dist:.4f}:")
    print(chunk[:500], "...")

Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Top retrieved chunks:

Distance 0.6448:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniv ...

Distance 0.6940:
hey joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made eight appearances in the Super Bowl. ...

Distance 0.8009:
The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). They defeated the Arizona Cardinals 49–15 in the NF

## Part 4: The Generator (GPT‑2)

We use a pre‑trained auto‑regressive language model (GPT‑2) to generate an answer conditioned on the retrieved context and the question.

**Input format** we will use:

Context:

chunk 1

chunk 2
...

Question: {user query}

Answer:

GPT‑2 will then continue the string.

### Questions 4
1. Why do we concatenate the context and the question?  
2. What are the limitations of using a base GPT‑2 model for knowledge‑intensive generation?

1. We concatenate the context and the question because GPT-2 has no separate input for external knowledge, it only sees one stream of text. By putting the retrieved chunks before the question, the relevant facts become part of the prompt, so when the model predicts the next tokens after `Answer:` it can attend to those facts instead of relying only on what it memorised during pre-training.

2. Base GPT-2 is only trained for next-token prediction and is not instruction-tuned, so instead of giving a short direct answer it just continues the prompt. Its knowledge is limited and outdated, and it tends to hallucinate. It also has a fairly small context window, so long retrieved contexts get truncated, and being a small model it is weak at actually reading the provided context and extracting the exact answer.

In [9]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

gpt2_model_name = "gpt2"  # You can also try "gpt2-medium" if resources allow.
tokenizer = GPT2Tokenizer.from_pretrained(gpt2_model_name)
model = GPT2LMHeadModel.from_pretrained(gpt2_model_name)
model.to(device)
model.eval()

# GPT-2 has no pad token by default; set it to eos for generation.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
# TODO: Implement `build_rag_prompt` and `generate_answer` for GPT-2.
# Concatenate retrieved context and question, then generate an answer.
# Expected output is kept below as a reference.

def build_rag_prompt(query, retrieved_chunks):
    context = retrieved_chunks[0]
    return (
        "Answer the question using only the context. Give a short answer.\n\n"
        "Context: The capital of France is Paris.\n"
        "Question: What is the capital of France?\n"
        "Answer: Paris\n\n"
        f"Context: {context}\n"
        f"Question: {query}\n"
        "Answer:"
    )


def generate_answer(query, retrieved_chunks, max_new_tokens=20):
    prompt = build_rag_prompt(query, retrieved_chunks)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return text.split("\n")[0].strip()   # keep only the first line



retrieved = [chunk for chunk, _ in retrieve(query, index, embed_model, chunks, top_k=3)]
answer = generate_answer(query, retrieved)

print("Question:", query)
print("Gold answer:", gold)
print("GPT-2 answer:", answer)

Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos
GPT-2 answer: The New England Patriots.


## Part 5: Putting It All Together – Stateless RAG

Now we combine the retriever and the generator into a single pipeline.

Given a question:
1. Retrieve top‑k chunks.
2. Concatenate them into a context string.
3. Feed context + question to the GPT‑2 generator.

Let’s implement a `RAGSystem` class that encapsulates this process.

### Question 5
This system answers each question independently. What problem arises when a user asks a follow‑up question like *“Tell me more about that”* or *“Multiply the previous answer by 2”*?

The problem is that this system is stateless, so each query only sees the current question and has no memory of what was asked or answered before.

The retriever embeds the vague follow-up on its own, and a question like "Tell me more about that" has no real topic in it, so the nearest chunks end up being almost random and the correct context is not retrieved. Also, even if some context is found, the generator cannot resolve the reference or reuse the earlier answer.


In [11]:
# TODO: Implement the stateless `RAGSystem` class.
# It should retrieve chunks, build the context, and generate an answer.
# Expected output is kept below as a reference.

class RAGSystem:
    def __init__(self, index, embed_model, chunks):
        self.index = index
        self.embed_model = embed_model
        self.chunks = chunks

    def query(self, question, top_k=3, max_new_tokens=40):
        retrieved = retrieve(question, self.index, self.embed_model, self.chunks, top_k)
        retrieved_chunks = [chunk for chunk, _ in retrieved]
        answer = generate_answer(question, retrieved_chunks, max_new_tokens=max_new_tokens)
        return {"answer": answer, "retrieved_chunks": retrieved_chunks}


rag = RAGSystem(index, embed_model, chunks)

sample = qa_dataset[0]
query = sample["question"]
gold = sample["answers"]["text"][0]
result = rag.query(query, top_k=3)

print("Question:", query)
print("Gold answer:", gold)

print("\nRetrieved chunks:")
for chunk in result["retrieved_chunks"]:
    print("-", chunk[:300], "...")

print("\nGPT-2 RAG answer:", result["answer"])

Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Retrieved chunks:
- Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super B ...
- hey joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made eight appearances in the Super Bowl. ...
- The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). They defeated the Arizona Cardinals 49–15 in the NFC Championship Game and advanced to their second Super Bowl appearance since the franchise was founded in 1995. The ...

GPT-2 RAG answer: The New England Patriots.


## Part 6: The Memory Problem

Let’s simulate a conversation where the second question depends on the first one.

**Stateless RAG will fail or become unstable** because each call is independent. The retriever only sees the latest question, so a follow-up such as *“Where was it played?”* or *“Which team represented the NFC in that game?”* is ambiguous unless the system remembers that the previous topic was **Super Bowl 50**.


In [12]:
q1 = qa_dataset[0]["question"]
ans1 = rag.query(q1, top_k=3, max_new_tokens=80)
print(f"Q: {q1}\nA: {ans1['answer']}\n")

# Follow-up that refers to the previous topic.
# Without memory, the phrase "that game" is ambiguous for the retriever.
q2 = "Which team represented the NFC in that game?"
ans2 = rag.query(q2, top_k=3, max_new_tokens=80)
print(f"Q: {q2}\nA: {ans2['answer']}\n")

# Notice: the second answer may be weaker because stateless RAG does not explicitly
# remember that the previous question was about Super Bowl 50.

Q: Which NFL team represented the AFC at Super Bowl 50?
A: The New England Patriots.

Q: Which team represented the NFC in that game?
A: The New Orleans Saints.



## Part 7: Adding Memory with LangChain

To enable multi-turn conversations, we need to **maintain a chat history** and use it to reformulate the current question into a standalone query.

In many LangChain tutorials, this is done with functions such as `create_history_aware_retriever` and `create_retrieval_chain`. However, these imports can break across different LangChain versions. To keep this notebook stable, we implement the same logic explicitly while still using LangChain for the FAISS vectorstore wrapper.

We will:

1. Wrap the same FAISS index and embedding model with LangChain’s `FAISS` vectorstore.
2. Store the conversation in a simple `chat_history` list.
3. Rewrite ambiguous follow-up questions into standalone questions.
4. Retrieve documents using the standalone question.
5. Generate the final answer using **FLAN-T5**, which is much better for instruction following than base GPT-2.

### Questions 6
1. Explain how a history-aware retriever works internally. What prompt or rewriting step does it use?  
2. Why can’t we simply pass the whole chat history to the retriever instead of generating a standalone question?


1. A history-aware retriever adds a small rewriting step before the actual retrieval. It first looks at the latest user question together with the chat history. If the question contains references, it uses a language model with a short prompt that says something like "given the conversation so far and the follow-up question, rewrite it into a standalone question that can be understood on its own".This standalone question is then embedded and sent to the retriever, so the missing context from earlier turns is put back into the query.

2. We can't pass the whole chat history to the retriever because the retriever turns its input into a single embedding and matches it by similarity. If we dump the entire conversation in, all the words from previous turns get mixed into one vector, which adds a lot of noise and pulls the search towards earlier topics instead of the current question. The result is an unfocused query. Generating a short standalone question keeps only the information that is actually needed to resolve the follow-up, so the embedding stays focused and retrieval stays precise.

In [13]:
# TODO: Create LangChain `Document` objects from chunks.
# Then build a FAISS vectorstore using HuggingFace embeddings.
# Expected output is kept below as a reference.

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS as LangchainFAISS

documents = [Document(page_content=chunk) for chunk in chunks]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = LangchainFAISS.from_documents(documents, embeddings)
print(f"Vectorstore contains {vectorstore.index.ntotal} documents.")

/tmp/ipykernel_31785/1149102685.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS as LangchainFAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectorstore contains 30 documents.


In [14]:
# TODO: Load `google/flan-t5-small` as an encoder-decoder model.
# Implement a small helper function to generate text with FLAN-T5.
# Expected output is kept below as a reference.

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_model_name = "google/flan-t5-small"
t5_tokenizer = AutoTokenizer.from_pretrained(t5_model_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_model_name).to(device)
t5_model.eval()


def flan_t5_generate(prompt, max_new_tokens=64, num_beams=4):
    inputs = t5_tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    with torch.no_grad():
        # beam search gives this small model cleaner, more reliable outputs
        output_ids = t5_model.generate(
            **inputs, max_new_tokens=max_new_tokens, num_beams=num_beams
        )
    return t5_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
# TODO: Implement helper functions for conversational RAG.
# Convert chat history to text and rewrite short follow-up questions.

import re

REFERENCE_WORDS = ["that game", "this game", "the game", "that team",
                   "that", "this", "those", "they", "them", "it"]


def _user_turns(chat_history):
    turns = []
    for message in chat_history:
        if isinstance(message, tuple):
            role, content = message
        else:
            role, content = message.type, message.content
        if role in ("human", "user"):
            turns.append(content)
    return turns


def format_chat_history(chat_history):
    lines = []
    for message in chat_history:
        if isinstance(message, tuple):
            role, content = message
        else:
            role, content = message.type, message.content
        speaker = "User" if role in ("human", "user") else "Assistant"
        lines.append(f"{speaker}: {content}")
    return "\n".join(lines)


def rewrite_question(question, chat_history):
    if not chat_history:
        return question
    if not re.search(r"\b(it|that|this|those|they|them|the game|that game)\b",
                     question, flags=re.IGNORECASE):
        return question


    prev_question = _user_turns(chat_history)[0]
    topic = flan_t5_generate(
        f"Extract the main subject of this question.\nQuestion: {prev_question}\nSubject:",
        max_new_tokens=16,
    )

    standalone = question
    for ref in REFERENCE_WORDS:
        replaced = re.sub(rf"\b{ref}\b", topic, standalone, count=1, flags=re.IGNORECASE)
        if replaced != standalone:
            return replaced
    return standalone

In [16]:
# TODO: Implement `SimpleConversationalRAG`.
# Steps: rewrite follow-up question, retrieve documents, generate answer.

class SimpleConversationalRAG:
    def __init__(self, vectorstore, k=3):
        self.vectorstore = vectorstore
        self.k = k

    def invoke(self, inputs):
        question = inputs["input"]
        chat_history = inputs.get("chat_history", [])

        standalone_question = rewrite_question(question, chat_history)

        docs = self.vectorstore.similarity_search(standalone_question, k=self.k)
        retrieved_context = "\n\n".join(doc.page_content for doc in docs)

        prompt = (
            "Answer the question using only the context below. "
            "Keep the answer short.\n\n"
            f"Context:\n{retrieved_context}\n\n"
            f"Question: {standalone_question}\n\n"
            "Answer:"
        )
        answer = flan_t5_generate(prompt, max_new_tokens=64)

        return {
            "input": question,
            "standalone_question": standalone_question,
            "retrieved_context": retrieved_context,
            "answer": answer,
        }


rag_chain_with_memory = SimpleConversationalRAG(vectorstore, k=3)

In [17]:
# Test the corrected memory-based RAG pipeline.
chat_history = []

question_1 = "What was Super Bowl 50?"
response_1 = rag_chain_with_memory.invoke({
    "input": question_1,
    "chat_history": chat_history
})

print("Question 1:", question_1)
print("Standalone question 1:", response_1["standalone_question"])
print("Answer 1:", response_1["answer"])

chat_history.append(("human", question_1))
chat_history.append(("ai", response_1["answer"]))

question_2 = "Where was it played?"
response_2 = rag_chain_with_memory.invoke({
    "input": question_2,
    "chat_history": chat_history
})

print("\nQuestion 2:", question_2)
print("Standalone question 2:", response_2["standalone_question"])
print("Answer 2:", response_2["answer"])
print("\nRetrieved context for turn 2:")
print(response_2["retrieved_context"][:1000], "...")


Question 1: What was Super Bowl 50?
Standalone question 1: What was Super Bowl 50?
Answer 1: an American football game to determine the champion of the National Football League (NFL) for the 2015 season

Question 2: Where was it played?
Standalone question 2: Where was Super Bowl 50 played?
Answer 2: San Francisco Bay Area at Santa Clara, California

Retrieved context for turn 2:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniv

On May 21, 2013, NFL owners at their spring meetings in Boston voted and awarded the game to Levi's Stadium. The $1

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []


def ask_question(query: str):
    response = rag_chain_with_memory.invoke({
        "input": query,
        "chat_history": chat_history
    })

    # Update chat history after each turn.
    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=response["answer"]))
    return response


# First turn: a real SQuAD-style question.
q1 = qa_dataset[0]["question"]
response1 = ask_question(q1)
print("User:", q1)
print("Standalone:", response1["standalone_question"])
print("Assistant:", response1["answer"])

# Follow-up question that depends on the previous topic.
q2 = "Which team represented the NFC in that game?"
response2 = ask_question(q2)
print("\nUser:", q2)
print("Standalone:", response2["standalone_question"])
print("Assistant:", response2["answer"])


User: Which NFL team represented the AFC at Super Bowl 50?
Standalone: Which NFL team represented the AFC at Super Bowl 50?
Assistant: Denver Broncos

User: Which team represented the NFC in that game?
Standalone: Which team represented the NFC in Super Bowl 50?
Assistant: Denver Broncos


### Explanation for Question 6

A history-aware retriever first checks whether the latest user question is self-contained. If the question contains references such as *it*, *that game*, or *the previous answer*, the system uses the chat history to rewrite it into a standalone query. For example:

```text
Chat history: What was Super Bowl 50?
Follow-up: Where was it played?
Standalone query: Where was Super Bowl 50 played?
```

The retriever should usually receive this standalone query rather than the whole chat history. Passing the entire chat history directly to the retriever can introduce irrelevant words from previous turns and make vector search less focused. A short standalone query preserves the missing context while keeping retrieval precise.

In this notebook, we implement the history-aware logic explicitly instead of relying on `langchain.chains`, because those imports are version-sensitive. The idea is the same: use memory to rewrite the query, retrieve relevant documents, then answer from the retrieved context.


## Part 8: RAG with an Encoder-Decoder Generator

So far, the retriever has been encoder-based, but the generator has been GPT‑2, which is a decoder-only language model.

In this part, we replace GPT‑2 with an encoder-decoder model, **google/flan-t5-small**. The retrieved context and the user question are passed to the encoder, and the decoder generates the answer.

### Question 7
Implement a RAG pipeline that uses the same retriever as before, but replaces GPT‑2 with an encoder-decoder model such as `google/flan-t5-small`.

1. Compare the GPT‑2-based RAG and the T5-based RAG on at least two questions.
2. Explain how the retrieved context is used differently in GPT‑2 and T5.
3. Why is T5 considered an encoder-decoder model, while GPT‑2 is decoder-only?

In [21]:
# TODO: Implement the T5-based RAG pipeline.
# Use the same retriever, but replace GPT-2 with FLAN-T5.
# Expected output is kept below as a reference.

def t5_rag_answer(query, top_k=3):
    retrieved_chunks = [chunk for chunk, _ in retrieve(query, index, embed_model, chunks, top_k)]
    context = "\n\n".join(retrieved_chunks)
    prompt = (
        "Answer the question using only the context below. Keep the answer short.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Answer:"
    )
    return flan_t5_generate(prompt, max_new_tokens=64), context


query = qa_dataset[0]["question"]
gold = qa_dataset[0]["answers"]["text"][0]
t5_answer, context = t5_rag_answer(query, top_k=3)

print("Question:", query)
print("Gold answer:", gold)
print("\nRetrieved context:")
print(context[:1000], "...")
print("\nT5 answer:")
print(t5_answer)

Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Retrieved context:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniv

hey joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made eight appearances in the Super Bowl.

The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). They defeated the Arizona Cardinals 49–15 in the NFC Championship Game and advanced to their second Super Bowl app

In [ ]:
# TODO: Compare GPT-2 RAG and T5 RAG on at least two questions.
# Use the same retrieved context for both models for a fair comparison.
# Expected output is kept below as a reference.

import pandas as pd

comparison = [
    (qa_dataset[0]["question"], "Denver Broncos"),
    ("What was the theme of Super Bowl 50?", '"golden anniversary"'),
]

records = []
for question, gold_answer in comparison:
    retrieved_chunks = [chunk for chunk, _ in retrieve(question, index, embed_model, chunks, top_k=3)]
    context = "\n\n".join(retrieved_chunks)

    gpt2_ans = generate_answer(question, retrieved_chunks, max_new_tokens=40)
    t5_prompt = (
        "Answer the question using only the context below. Keep the answer short.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )
    t5_ans = flan_t5_generate(t5_prompt, max_new_tokens=64)

    records.append({
        "Question": question,
        "Gold Answer": gold_answer,
        "GPT-2 RAG Answer": gpt2_ans,
        "T5 Encoder-Decoder RAG Answer": t5_ans,
        "context": context,
    })

df = pd.DataFrame(records)[
    ["Question", "Gold Answer", "GPT-2 RAG Answer", "T5 Encoder-Decoder RAG Answer"]
]
print(df)

for r in records:
    print("\n" + "=" * 100)
    print("Question:", r["Question"])
    print("Gold answer:", r["Gold Answer"])
    print("\nGPT-2 RAG Answer:\n" + r["GPT-2 RAG Answer"])
    print("\nT5 Encoder-Decoder RAG Answer:\n" + r["T5 Encoder-Decoder RAG Answer"])
    print("\nRetrieved Context:\n" + r["context"][:1200], "...")

                                            Question           Gold Answer  \
0  Which NFL team represented the AFC at Super Bo...        Denver Broncos   
1               What was the theme of Super Bowl 50?  "golden anniversary"   

                                    GPT-2 RAG Answer  \
0                          The New England Patriots.   
1  The theme of Super Bowl 50 was "The Greatest G...   

                       T5 Encoder-Decoder RAG Answer  
0                                     Denver Broncos  
1  gold-themed initiatives, as well as temporaril...  

Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

GPT-2 RAG Answer:
The New England Patriots.

T5 Encoder-Decoder RAG Answer:
Denver Broncos

Retrieved Context:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conf

### Explanation for Question 7

In both systems, the retriever is exactly the same. The question is embedded using the SentenceTransformer model, and FAISS retrieves the top-k most relevant chunks from the real SQuAD context corpus.

The main difference is the generator architecture.

In the **GPT‑2-based RAG system**, the retrieved context, the question, and the `Answer:` prefix are concatenated into one prompt. GPT‑2 is a **decoder-only** model, so it generates the answer by continuing this prompt from left to right. During generation, each new token can attend only to previous tokens, including the retrieved context and the question.

In the **T5-based RAG system**, the retrieved context and the question are passed to the **encoder**. The encoder reads the full input sequence and produces contextual representations. Then the **decoder** generates the answer autoregressively while attending to the encoder outputs through cross-attention. Therefore, the retrieved context is used as an encoded source sequence rather than only as a text prefix.

T5 is considered an **encoder-decoder** model because it has two separate Transformer components: an encoder for processing the input text and a decoder for generating the output text. GPT‑2 is **decoder-only** because it only contains the autoregressive Transformer decoder stack and generates text by predicting the next token from the previous tokens.

In practice, FLAN‑T5 usually gives cleaner answers for this task than base GPT‑2 because FLAN‑T5 is instruction-tuned and better aligned with question answering. Base GPT‑2 is mainly trained for next-token prediction, so it may continue the prompt instead of directly answering the question.

## Summary

You have:
- Built a **stateless RAG system** from real SQuAD contexts, embeddings, FAISS, and GPT‑2.
- Observed its weakness on follow-up questions.
- Used a LangChain FAISS vectorstore and an explicit **history-aware query rewriting** step to add conversational memory.
- Replaced the unstable GPT‑2 memory generator with **FLAN‑T5** for reformulation and answering.
- Replaced the decoder-only GPT‑2 generator with an **encoder-decoder FLAN‑T5** generator.
- Compared GPT‑2-based RAG and T5-based RAG on the same retrieved contexts.

### Final Theoretical Questions
1. Compare the stateless and conversational RAG pipelines. What are the main differences in the retrieval step?
2. In the conversational version, where is the ‘memory’ actually stored?
3. How could you extend this system to handle **multiple users**? What would you need to change?
4. Suppose you wanted to replace GPT‑2 with a model that does not support the “reformulate a standalone question” task well. What alternatives could you explore?
5. Why can an encoder-decoder model such as T5 be more suitable for question answering than a base decoder-only model such as GPT‑2?

### References
- [SQuAD: 100,000+ Questions for Machine Comprehension of Text](https://rajpurkar.github.io/SQuAD-explorer/)
- [Lewis et al., Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (2020)](https://arxiv.org/abs/2005.11401)
- [FAISS library](https://github.com/facebookresearch/faiss)
- [FLAN-T5 model family](https://huggingface.co/google/flan-t5-small)


### Answers to the Theoretical Questions

1. In the stateless pipeline the retriever uses the raw current question directly to search the index, so a follow-up with pronouns retrieves the wrong chunks. In the conversational pipeline there is an extra step before retrieval: the follow-up is first rewritten into a standalone question using the chat history, and the retriever searches with that resolved question. So the main difference is the input to the retriever, the raw question versus a history-aware standalone question.

2. The memory is stored in the `chat_history` list that we keep in the application, not inside the model or the vector store. The models are stateless, we just pass the history into each call. The vector store only holds the document chunks, it does not remember the conversation.

3. To handle multiple users we would keep a separate chat history per user, for example a dictionary keyed by a user or session id, and load the correct history on each request. The embedding model, the FAISS index and the generator can stay shared. If different users have private documents, we would also need a separate vector store or a metadata/namespace filter per user, plus some session management to isolate their conversations.

4. If the generator cannot reformulate the question well, we could use a separate model only for the rewriting step, such as a slightly larger instruction-tuned model like flan-t5-base, or a dedicated query-rewriting model. Other options are to resolve the references with a rule-based / coreference step, to simply concatenate the recent chat history with the question and retrieve with that, or to fine-tune a small model specifically on the query-rewriting task.

5. An encoder-decoder model like T5 has a bidirectional encoder that reads the whole context and question at once (each token attends both left and right), so it builds a rich understanding of the passage, and a decoder that produces the answer through cross-attention to those encoded representations. It is also trained in a text-to-text way (and FLAN-T5 is instruction-tuned), so it directly maps input to answer. A decoder-only model like GPT-2 only attends left-to-right and is trained to continue text, so it reads the context less effectively and tends to keep writing instead of giving a direct answer.